In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

metadata = pd.read_csv("metadata/metadata_Instance_events_10k.csv")


In [2]:
print(metadata.shape)

num_earthquakes = metadata["source_id"].nunique()
print(num_earthquakes)


(10000, 115)
300


In [23]:
metadata["source_origin_time"] = pd.to_datetime(metadata["source_origin_time"])
metadata["year"] = metadata["source_origin_time"].dt.year
print(metadata["year"].min())
print(metadata["year"].max())

2012
2016


In [21]:
metadata_pre2018 = metadata[metadata["year"] < 2018]
metadata_after2018 = metadata[metadata["year"] >= 2018]

In [22]:
train_ids = metadata_pre2018["source_id"].unique()

event_ids = metadata_after2018["source_id"].unique()
val_ids, test_ids = train_test_split(
    event_ids,
    test_size=0.50,
    random_state=42
)
print(len(train_ids), len(val_ids), len(test_ids))

ValueError: With n_samples=0, test_size=0.5 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [4]:
train_df = metadata[metadata["source_id"].isin(train_ids)].copy()
val_df   = metadata[metadata["source_id"].isin(val_ids)].copy()
test_df  = metadata[metadata["source_id"].isin(test_ids)].copy()

print(len(train_df))
print(len(val_df))
print(len(test_df))

7326
1306
1368


In [5]:
required_columns = [
    "source_magnitude",
    "trace_P_arrival_sample"
]

train_df = train_df.dropna(subset=required_columns)
val_df   = val_df.dropna(subset=required_columns)
test_df  = test_df.dropna(subset=required_columns)

print(len(train_df))
print(len(val_df))
print(len(test_df))

7326
1306
1368


In [6]:
import h5py as h5

In [7]:
waveform_file = h5.File ("data/Instance_events_counts_10k.hdf5","r")
print(waveform_file["data_format"].keys())
print(waveform_file["data"]["11030611.IV.OFFI..HH"])



<KeysViewHDF5 ['component_order', 'dimension_order', 'instrument_response', 'measurement', 'unit']>
<HDF5 dataset "11030611.IV.OFFI..HH": shape (3, 12000), type "<i4">


In [8]:
trace_name = "11030611.IV.OFFI..HH"
row = train_df[train_df["trace_name"] == trace_name].iloc[0]
print(row)
waveform = waveform_file["data"][trace_name][:]
print("Trace:", trace_name)
print("Waveform shape:", waveform.shape)
print("Magnitude:", row["source_magnitude"])
print("P arrival:", row["trace_P_arrival_sample"])

source_id                        11030611
station_network_code                   IV
station_code                         OFFI
station_location_code                 NaN
station_channels                       HH
                                  ...    
trace_EQT_number_detections           1.0
trace_EQT_P_number                    1.0
trace_EQT_S_number                    1.0
trace_deconvolved_units               mps
source_type                    earthquake
Name: 0, Length: 115, dtype: object
Trace: 11030611.IV.OFFI..HH
Waveform shape: (3, 12000)
Magnitude: 2.5
P arrival: 1735


0    2016-12-04T15:34:52.05Z
1    2016-12-04T15:34:52.05Z
2    2016-12-04T15:34:52.05Z
3    2016-12-04T15:34:52.05Z
4    2016-12-04T15:34:52.05Z
Name: source_origin_time, dtype: str


In [9]:
row = train_df.iloc[0]
trace_name = row["trace_name"]
p_arrival = int(row["trace_P_arrival_sample"])
magnitude = row["source_magnitude"]

waveform = waveform_file["data"][trace_name][:]

print("Full waveform:", waveform.shape)
print("P arrival:", p_arrival)
print("Magnitude:", magnitude)

waveform_3s = waveform[:, p_arrival:p_arrival + 300]

print("3 second waveform:", waveform_3s.shape)

Full waveform: (3, 12000)
P arrival: 1735
Magnitude: 2.5
3 second waveform: (3, 300)


In [10]:

waveforms_3s = []
magnitudes = []

for i in range (len(train_df)):
    row = train_df.iloc[i]
    trace_name = row["trace_name"]
    p_arrival = int(row["trace_P_arrival_sample"])
    magnitude = row["source_magnitude"]
    magnitudes.append(magnitude)
    waveform = waveform_file["data"][trace_name][:]
    


    waveform_3s = waveform[:, p_arrival:p_arrival + 300]
    waveforms_3s.append(waveform_3s)




In [11]:
import numpy as np

print(np.isnan(waveform_3s).any())
print(np.isinf(waveform_3s).any())

False
False


In [12]:
print(waveform_3s.mean(axis=1))
print(waveform_3s.std(axis=1))
print(waveform_3s.min(axis=1))
print(waveform_3s.max(axis=1))

[-12.16        17.72666667  40.52333333]
[579.53724735 301.29880842 784.78628266]
[-1894  -721 -1641]
[1031  727 3079]


In [13]:
waveforms_3s = np.array(waveforms_3s)
print(waveforms_3s.shape)


(7326, 3, 300)


In [14]:
import numpy as np
waveforms_3s = np.stack(waveforms_3s)   # shape: (N, 3, 300)

mean = waveforms_3s.mean(axis=(0, 2), keepdims=True)
std = waveforms_3s.std(axis=(0, 2), keepdims=True)

In [15]:
for w in waveforms_3s:
    if w.shape != (3, 300):
        print(w.shape)

In [16]:
train_df.to_csv("train_metadata.csv", index=False)
val_df.to_csv("val_metadata.csv", index=False)
test_df.to_csv("test_metadata.csv", index=False)
np.save("train_mean.npy", mean)
np.save("train_std.npy", std)